# Introduction

## Purpose

_Created by Romain Molins, with support from Philip Bradstock from Bitbloom_

This notebook has been developed to generate long-term Production Time Series from a few years (minimum one) of OYA PTS

On Knowledge Share:


## Version History

- 1.0: Base version

## Loading Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
from IPython.display import FileLink
import re
import brightwind as bw

import scipy
from scipy.stats import loguniform, uniform, randint
from functools import partial
from sklearn import metrics
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import PartialDependenceDisplay

from sklearn.model_selection import cross_val_predict, KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

from cmethods import adjust

## Functions

In [ ]:
_NanosecondsInDay = 24 * 3600 * 10**9
_NanosecondsInYear = 365.2425 * _NanosecondsInDay

#do you want the graphsd in 3.2 to be enabled or not
inspection = False

def _get_radians(timestamps: pd.DatetimeIndex, period):
    if period == 'day':
        return 2 * np.pi * np.mod(timestamps.astype(np.int64) / _NanosecondsInDay, 1.0)
    if period == 'year':
        return 2 * np.pi * np.mod(timestamps.astype(np.int64) / _NanosecondsInYear, 1.0)


def get_time_comps(data, period):
    data_radians = _get_radians(data, period)
    return np.cos(data_radians), np.sin(data_radians)

# is added below as it corresponds to the wind speed column, so we focus on wind speed metric. This is dirty, should be cleaned later
def binned_error(y_true, y_pred, bins_0=None):
    # Precompute bins if not provided
    if bins_0 is None:
        bins_0 = np.arange(0.0, y_true.max() , 1)
    # Compute histograms
    pred_freq, _ = np.histogram(y_pred, bins=bins_0)
    train_freq, _ = np.histogram(y_true, bins=bins_0)
    # Normalize (avoid division by zero)
    pred_sum = pred_freq.sum()
    train_sum = train_freq.sum()
    if pred_sum == 0 or train_sum == 0:
        return np.nan
    pred_freq = pred_freq / pred_sum
    train_freq = train_freq / train_sum
    # Use mean_absolute_error directly (train_freq.mean() is 1)
    return mean_absolute_error(train_freq, pred_freq)

#this one combines the previous error with the RMSE classic error, summing them
def overall_score(y_true, y_pred):
    bin_error = binned_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred)) / y_true.max()
    return bin_error + rmse

## User Input

In [ ]:
#Define timezone
time_zone_reference = 'Etc/GMT+2'
time_zone_site = 'Etc/GMT+2'
time_zone_output = 'Etc/GMT+2'

project = 'Waaihoek'

# path of input reference timeseries file that will be used as X
reference_file = "../data/vortex.times.864091.20y 100m UTC+02.0 ERA5.txt"

#reference data period
reference_start_date = '2005-01-01 00:00'
reference_end_date = '2025-05-30 23:50'

# path of input onsite timeseries file that will be used as Y.
onsite_file = "../data/SA014_02_clean_cmbAno_2025 100m.txt"

#training period - usually the period of the OYA PTS
start_training = '2016-01-01 00:00'
end_training = '2016-12-31 23:50'

#testing period - usually the period of the OYA PTS
start_testing = '2017-01-01 00:00'
end_testing = '2017-12-31 23:50'

#prediction period - or long-term period required for the output - cannot be great than the vortex dates
start_pred_LT = '2005-01-01 00:00'
end_pred_LT = '2025-05-30 23:50'

#reference nodes
v_wsnode = 'M(m/s)'
v_wdnode = 'D(deg)'
v_adnode = 'De(k/m3)'
v_RInode = 'RiNumber'
v_tnode = 'T(C)'
v_pnode = 'PRE(hPa)'

#onsite nodes
o_wsnode = 'Speed 100m cmb [m/s]'
o_wdnode = 'Wind Direction 97.90m S avg [°]'


# Loading data

## Load Vortex

In [ ]:
#raw Vortex SERIES format below
vortex_data = pd.read_fwf(reference_file, skiprows=3,parse_dates=[['YYYYMMDD', 'HHMM']], index_col=0)
vortex_data.index = vortex_data.index.tz_localize(time_zone_reference)

#go from polar to cartesian
vortex_data['cos_u'] = np.cos(np.deg2rad(vortex_data[v_wdnode])) * vortex_data[v_wsnode]
vortex_data['sin_u'] = np.sin(np.deg2rad(vortex_data[v_wdnode])) * vortex_data[v_wsnode]

#then conver hour and month to polar domain, to facilitate ML training (as months and hours are cyclical)
year_cos, year_sin = get_time_comps(vortex_data.index, 'year')
vortex_data['time_of_year_cos'] = year_cos
vortex_data['time_of_year_sin'] = year_sin

day_cos, day_sin = get_time_comps(vortex_data.index, 'day')
vortex_data['time_of_day_cos'] = day_cos
vortex_data['time_of_day_sin'] = day_sin

#trim to start/end
vortex_data = vortex_data.loc[reference_start_date:reference_end_date,:].copy()
vortex_data

## Load site data

In [ ]:
#onsite data
onsite_data = pd.read_csv(onsite_file, sep=",", parse_dates=True, index_col=0, skiprows=14, dayfirst=False)
onsite_data.index = onsite_data.index.tz_localize(time_zone_site)

#trim to start/end
#onsite_data = onsite_data.loc[start_training:end_training,:].copy()
onsite_data['o_cos_u'] =  np.cos(np.deg2rad(onsite_data[o_wdnode])) * onsite_data[o_wsnode]
onsite_data['o_sin_u'] =  np.sin(np.deg2rad(onsite_data[o_wdnode])) * onsite_data[o_wsnode]

onsite_data

# ML Model

## Baseline Model

In [ ]:
#columns used for Vortex (X) model
columnsX = ['cos_u','sin_u', v_tnode, v_pnode, 'time_of_day_cos', 'time_of_day_sin','time_of_year_cos', 'time_of_year_sin']
#columns for postprocessing
columnsX_ps = [v_wsnode, v_wdnode]

#columns for training
columnsY = ['o_cos_u', 'o_sin_u']
#columns for postprocessing
columnsY_ps = [o_wsnode, o_wdnode]


#we merge to align on the same timestamps and avoid gaps
data_train = vortex_data.loc[start_training:end_training,columnsX+columnsX_ps].merge(onsite_data.loc[start_training:end_training,columnsY+columnsY_ps],how='outer',left_index=True, right_index=True)

#remove empty rows with missing values - no nan can be in the training data
data_train = data_train.dropna(axis=0, how='any')

#define X and y
X_train = data_train.loc[start_training:end_training,columnsX]
y_train = data_train.loc[start_training:end_training,columnsY]


In [ ]:
columnsX2 = [v_wsnode, v_wdnode, v_tnode, v_pnode]
columnsY2 = [o_wsnode, o_wdnode]


In [ ]:
# set up correlation
mlp = bw.Correl.MultiLayerPerceptron(
    ref_spd=vortex_data.loc[:,columnsX2],
    target_spd=onsite_data.loc[start_training:end_training,columnsY2],
    averaging_prd='10min',
    alpha=0.1,
    ref_spd_col=v_wsnode,
    tar_spd_col=o_wsnode,
    ref_dir_col=v_wdnode,
    tar_dir_col=o_wdnode
    )

In [ ]:
mlp.run()

In [ ]:
columnsX3 = list(mlp.ref_spd.columns)
columnsX3.remove(mlp.ref_spd_col)
columnsX3.remove(mlp.ref_dir_col)
columnsX3

In [ ]:
#if ext_input is not None, no splicing done
y_pred = mlp.synthesize(ext_input=mlp.ref_spd[columnsX3])

In [ ]:
y_pred

In [ ]:
#post process to get wind speed and directions back
y_pred.loc[start_training:end_training,:]

In [ ]:
y_pred_training = y_pred.loc[start_training:end_training,:].copy()
y_pred_training

In [ ]:
#just some postprocessing to convert the output to dataframe and CSV output
y_pred_testing = y_pred.loc[start_testing:end_testing,:]
y_pred_testing

In [ ]:
y_test = onsite_data.loc[start_testing:end_testing,columnsY_ps].copy()
y_test

In [ ]:
test_merge

In [ ]:
test_merge

In [ ]:
#remove NANs!
test_merge = y_pred_testing.merge(y_test, left_index=True, right_index=True,suffixes= ('', '_Original'))
test_merge.dropna(inplace=True)
y_pred_testing = test_merge[[o_wsnode+'_Synthesized', o_wdnode+'_Synthesized']].copy()
y_pred_testing.columns = y_pred_testing.columns.str.removesuffix('_Synthesized')
y_test = test_merge[[o_wsnode, o_wdnode]].copy()
y_test

In [ ]:
y_pred_testing

## Scoring

In [ ]:
#we give example of both error values using the previous baseline models - wind speed
overall_score_best = overall_score(y_test[o_wsnode].to_numpy(), y_pred_testing[o_wsnode].to_numpy())
binned_error_best = binned_error(y_test[o_wsnode].to_numpy(), y_pred_testing[o_wsnode].to_numpy())
best_score_best = 100 * sqrt(mean_squared_error(y_test[o_wsnode], y_pred_testing[o_wsnode])) / y_test[o_wsnode].max()
print(f'Wind speed score is {best_score_best:.2f}% ')
print(f'Wind speed binned error is {100*binned_error_best:.2f}% ')
print(f'Wind speed overall error is  {100*overall_score_best:.2f}% ')

In [ ]:
#we give example of both error values using the previous baseline models - wind direction
overall_score_best = overall_score(y_test[o_wdnode].to_numpy(), y_pred_testing[o_wdnode].to_numpy())
binned_error_best = binned_error(y_test[o_wdnode].to_numpy(), y_pred_testing[o_wdnode].to_numpy())
best_score_best = 100 * sqrt(mean_squared_error(y_test[o_wdnode], y_pred_testing[o_wdnode])) / y_test[o_wdnode].max()
print(f'Wind speed score is {best_score_best:.2f}% ')
print(f'Wind speed binned error is {100*binned_error_best:.2f}% ')
print(f'Wind speed overall error is  {100*overall_score_best:.2f}% ')

## LT Prediction

In [ ]:
WS_lt

In [ ]:
y_pred_LT

In [ ]:
#and we predict it
X_pred_LT = vortex_data.loc[start_pred_LT:end_pred_LT,columnsX]
X_pred_LT.replace([np.inf, -np.inf], 0, inplace=True)

#X_pred_LT = np.nan_to_num(X_pred_LT) #to avoid nans at beginning and end due to t+1/t-1
y_pred_LT = y_pred.copy()

#just some postprocessing to convert the output to dataframe and CSV output
WS_lt = y_pred_LT.loc[start_pred_LT:end_pred_LT,:]
WS_lt.columns = WS_lt.columns.str.removesuffix('_Synthesized')
WS_lt.index.name = onsite_data.index.name
WS_lt

# Validation

In the previous 'Scoring' sectiosn we focused on ML metrics to define the best model 
Here as complement we validate the model and ensure that the LT PTS distribution, hourly/monthly/yearly profiles and frequency patterns makes sense compared to the OYA PTS and Vortex data

In [ ]:
data_predict_train = WS_lt.loc[start_training:end_training,columnsY_ps].merge(onsite_data.loc[start_training:end_training,columnsY_ps],how='inner',left_index=True, right_index=True,suffixes= ('_Predicted', '_Original') )
data_predict_train.dropna(inplace=True)
y_predict_train = data_predict_train.loc[start_training:end_training,[l + '_Predicted' for l in columnsY_ps]].copy()
y_predict_train

In [ ]:
onsite_data.loc[start_testing:end_testing,columnsY_ps]

In [ ]:
data_predict_test = WS_lt.loc[start_testing:end_testing,columnsY_ps].merge(onsite_data.loc[start_testing:end_testing,columnsY_ps],how='inner',left_index=True, right_index=True,suffixes= ('_Predicted', '_Original') )
data_predict_test.dropna(inplace=True)
y_predict_test = data_predict_test.loc[start_testing:end_testing,[l + '_Predicted' for l in columnsY_ps]].copy()
y_predict_test

## Scatter Plot

In [ ]:
#we calculate back the wind speed if missing
data_train[v_wsnode] = np.sqrt(data_train['cos_u']**2 + data_train['sin_u']**2) 

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(18, 8))

ylim = (0, 30)

# 1st plot: ws vs y_train
#r2_score is not equal to pearson squared for any fit that is non linear!
#r2_1 = r2_score(data_train['ws scaled'], data_train[o_wsnode])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_train[v_wsnode], data_train[o_wsnode])
r2_1 = r_value**2

ax[0].scatter(data_train[v_wsnode], data_train[o_wsnode], marker='.', alpha=0.5)
ax[0].set_xlabel('Vortex Wind Speed [m/s] (training period)')
ax[0].set_ylabel('Onsite Wind speed [m/s] (training period)')
ax[0].set_ylim(*ylim)
ax[0].set_title(f'R² = {r2_1:.2f}')

# 2nd plot: ws vs best_y_pred
#r2_2 = r2_score(data_predict_train.iloc[:,0], data_predict_train.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_train[o_wsnode+'_Original'], data_predict_train[o_wsnode+'_Predicted'])
r2_2 = r_value**2

ax[1].scatter(data_predict_train[o_wsnode+'_Original'],data_predict_train[o_wsnode+'_Predicted'], marker='.', alpha=0.5)
ax[1].set_xlabel('Onsite Wind Speed [m/s] (training period)')
ax[1].set_ylabel('Predicted wind speed [m/s] (training period)')
ax[1].set_ylim(*ylim)
ax[1].set_title(f'R² = {r2_2:.2f}')


# 3nd plot: test period
#r2_3 = r2_score(data_predict_test.iloc[:,0], data_predict_test.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_test[o_wsnode+'_Original'], data_predict_test[o_wsnode+'_Predicted'])
r2_3 = r_value**2

ax[2].scatter(data_predict_test[o_wsnode+'_Original'], data_predict_test[o_wsnode+'_Predicted'], marker='.', alpha=0.5)
ax[2].set_xlabel('Vortex Wind Speed [m/s] (testing period)')
ax[2].set_ylabel('Predicted wind speed [m/s] (testing period)')
ax[2].set_ylim(*ylim)
ax[2].set_title(f'R² = {r2_3:.2f}')


plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(18, 8))

ylim = (0, 360)

# 1st plot: ws vs y_train
#r2_score is not equal to pearson squared for any fit that is non linear!
#r2_1 = r2_score(data_train['ws scaled'], data_train[o_wdnode])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_train[v_wdnode], data_train[o_wdnode])
r2_1 = r_value**2

ax[0].scatter(data_train[v_wdnode], data_train[o_wdnode], marker='.', alpha=0.5)
ax[0].set_xlabel('Vortex Wind Direction [°] (training period)')
ax[0].set_ylabel('Onsite Wind Direction [°] (training period)')
ax[0].set_ylim(*ylim)
ax[0].set_title(f'R² = {r2_1:.2f}')

# 2nd plot: ws vs best_y_pred
#r2_2 = r2_score(data_predict_train.iloc[:,0], data_predict_train.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_train[o_wdnode+'_Original'], data_predict_train[o_wdnode+'_Predicted'])
r2_2 = r_value**2

ax[1].scatter(data_predict_train[o_wdnode+'_Original'],data_predict_train[o_wdnode+'_Predicted'], marker='.', alpha=0.5)
ax[1].set_xlabel('Onsite Wind Direction [°] (training period)')
ax[1].set_ylabel('Predicted Wind Direction [°] (training period)')
ax[1].set_ylim(*ylim)
ax[1].set_title(f'R² = {r2_2:.2f}')


# 3nd plot: test period
#r2_3 = r2_score(data_predict_test.iloc[:,0], data_predict_test.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_test[o_wdnode+'_Original'], data_predict_test[o_wdnode+'_Predicted'])
r2_3 = r_value**2

ax[2].scatter(data_predict_test[o_wdnode+'_Original'], data_predict_test[o_wdnode+'_Predicted'], marker='.', alpha=0.5)
ax[2].set_xlabel('Vortex Wind Direction [°] (testing period)')
ax[2].set_ylabel('Predicted Wind Direction [°] (testing period)')
ax[2].set_ylim(*ylim)
ax[2].set_title(f'R² = {r2_3:.2f}')


plt.tight_layout()
plt.show()

## Histogram

In [ ]:
#Wind speed
fig, ax = plt.subplots(figsize=(18, 8))
bins_0 = np.arange(0.0, 30,1)
ax.hist(onsite_data[o_wsnode],bins=bins_0, density = True, histtype='step', lw = 2, label='On-Site WS data (Training)')
ax.hist(y_predict_train[o_wsnode+'_Predicted'],bins=bins_0, density = True, histtype='step', lw = 2, label='best_y_pred (training)')
ax.hist(WS_lt[o_wsnode],bins=bins_0, density = True, histtype='step', lw = 2, label='y_pred_LT (long-term)')

handles, labels = ax.get_legend_handles_labels()
lgd = fig.legend(handles, labels)
#plt.ylim(0,0.00002)
plt.xlabel('Wind speed [m/s]')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Wind speed
fig, ax = plt.subplots(figsize=(18, 8))
bins_0 = np.arange(0.0, 360,10)
ax.hist(onsite_data[o_wdnode],bins=bins_0, density = True, histtype='step', lw = 2, label='On-Site WD data (Training)')
ax.hist(y_predict_train[o_wdnode+'_Predicted'],bins=bins_0, density = True, histtype='step', lw = 2, label='best_y_pred (training)')
ax.hist(WS_lt[o_wdnode],bins=bins_0, density = True, histtype='step', lw = 2, label='y_pred_LT (long-term)')

handles, labels = ax.get_legend_handles_labels()
lgd = fig.legend(handles, labels)
#plt.ylim(0,0.00002)
plt.xlabel('Wind direction [°]')
plt.ylabel('Frequency')
plt.show()

## Monthly/hourly profiles

In [ ]:
# binning of original PTS
# Visualise monthly and hourly wind speed profiles

# Use only wind speed columns for plotting (e.g., o_wsnode for onsite, 'M(m/s)' for reference, and WS_lt for prediction)
onsite_month = onsite_data.groupby(onsite_data.index.month).mean()[o_wsnode]
onsite_hour = onsite_data.groupby(onsite_data.index.hour).mean()[o_wsnode]
reference_month = vortex_data.groupby(vortex_data.index.month).mean()[v_wsnode]
reference_hour = vortex_data.groupby(vortex_data.index.hour).mean()[v_wsnode]
WS_lt_month = WS_lt.groupby(WS_lt.index.month).mean()[o_wsnode]
WS_lt_hour = WS_lt.groupby(WS_lt.index.hour).mean()[o_wsnode]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(18, 8))

# Monthly profile
ax[0].plot(WS_lt_month.index, WS_lt_month.values, 'k-', label='Long Term Wind Speed (Prediction)')
ax[0].plot(onsite_month.index, onsite_month.values, 'r-', lw=2, label='On-Site WS data (Training)')
ax[0].set_xlabel('Month')
ax[0].set_xticks(np.arange(1, 13, 1))
ax[0].set_ylabel('Wind Speed [m/s]')
ax[0].set_title('Monthly Wind Speed Profile')

# Hourly profile
ax[1].plot(WS_lt_hour.index, WS_lt_hour.values, 'k-', label='Long Term Wind Speed (Prediction)')
ax[1].plot(onsite_hour.index, onsite_hour.values, 'r-', lw=2, label='On-Site WS data (Training)')
ax[1].set_xlabel('Hour of Day')
ax[1].set_xticks(np.arange(0, 24, 3))
ax[1].set_ylabel('Wind Speed [m/s]')
ax[1].set_title('Hourly Wind Speed Profile')

# Combine legends
handles1, labels1 = ax[0].get_legend_handles_labels()
plt.legend(handles1, labels1 , loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

## Annual Profile

In [ ]:
# Calculate annual mean wind speed for actual (onsite), predicted (WS_lt), and reference (Vortex) data
onsite_annual = onsite_data.groupby(onsite_data.index.year)[o_wsnode].mean()
predicted_annual = WS_lt.groupby(WS_lt.index.year)[o_wsnode].mean()
reference_annual = vortex_data.groupby(vortex_data.index.year)[v_wsnode].mean()

# Plot all three on the same figure
fig, ax = plt.subplots(figsize=(18, 8))
ax.plot(onsite_annual.index, onsite_annual.values, 'r-o', label='On-Site Wind Speed (Actual)')
ax.plot(predicted_annual.index, predicted_annual.values, 'k-o', label='Long Term Wind Speed (Predicted)')
ax.plot(reference_annual.index, reference_annual.values, 'g-o', label='Reference Wind Speed (Reference)')
ax.set_xlabel('Year')
ax.set_ylabel('Wind Speed [m/s]')
ax.set_title('Annual Wind Speed Profile: Actual vs Predicted vs Reference')
ax.legend()
plt.show()


## FFT

In [ ]:
# Number of samplepoints
N2 = len(onsite_data)
N = len(WS_lt)

# sample spacing (in seconds)
T = 3600
# Calculate the FFT frequency step robustly
if WS_lt.index.freq is not None:
    nb_sec_freq = WS_lt.index.freq.delta.total_seconds()
else:
    # fallback: infer frequency from index if freq is None
    nb_sec_freq = (WS_lt.index[1] - WS_lt.index[0]).total_seconds()

#FFT of Synthesised data
y = WS_lt.iloc[:,0]
yf = scipy.fftpack.fft(y.values)
xf = np.linspace(0.0, 1.0/(2.0*T), int(N/2))
xf_hour = 1/xf/3600/24/(nb_sec_freq/100)

#FFT of Measured data
y2 = onsite_data[o_wsnode]
y2 = y2.fillna(0) #Replacing NaN by 0s
yf2 = scipy.fftpack.fft(y2.values)
xf2 = np.linspace(0.0, 1.0/(2.0*T), int(N2/2))
xf_hour2 = 1/xf2/3600/24/6

#Plot
fig, ax = plt.subplots(figsize=(18, 10))
ax.plot(xf_hour, 2.0/N * np.abs(yf[:N//2]), color = 'black', label='Long Term PTS (Prediction)' )
ax.plot(xf_hour2, 2.0/N2 * np.abs(yf2[:N2//2]), color = 'red',linestyle=':', label='OYA PTS (Training)')
ax.set_xlim(0,10)
ax.set_xticks(np.arange(0, 10, step=1))
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles, labels)
ax.set_xlabel('Days')
ax.set_ylabel('Amplitude')

plt.show()

## LT Scaling Factor

In [ ]:
print(f'Mean wind speed of onsite data during training period is {onsite_data.loc[start_training:end_training,o_wsnode].mean():.2f} m/s ')
print(f'Mean wind speed of predicted data during training period is {y_predict_train.loc[start_training:end_training,o_wsnode+'_Predicted'].mean():.2f} m/s ')
print(f'Mean wind speed of predicted data during long-term period is {WS_lt.loc[:,o_wsnode].mean():.2f} m/s ')
print(f'LT scaling factor is {100*((WS_lt.loc[:,o_wsnode].mean()/y_predict_train.loc[start_training:end_training,o_wsnode+'_Predicted'].mean())-1):.2f}% ')


# Quantile Mapping

## Empirical Quantile Mapping Model

### Wind speed

In [ ]:
#copy df for EQM
onsite_data_eqm = onsite_data.loc[start_training:end_training,:].copy()
y_predict_train_eqm = y_predict_train.copy()
WS_lt_eqm = WS_lt.copy()

#edit index name
onsite_data_eqm.index.names = ['time1']
y_predict_train_eqm.index.names = ['time2']
WS_lt_eqm.index.names = ['time3']

#edit index type
onsite_data_eqm.index = onsite_data_eqm.index.tz_localize(None)
y_predict_train_eqm.index = y_predict_train_eqm.index.tz_localize(None)
WS_lt_eqm.index = WS_lt_eqm.index.tz_localize(None)

#obsh = observation for the historical period
#simh = simulation for the historical period
#simp = observation for the future period

print("obsh is from " + str(onsite_data_eqm.index.min()) + " to " + str(onsite_data_eqm.index.max()))
print("simh is from " + str(y_predict_train_eqm.index.min()) + " to " + str(y_predict_train_eqm.index.max()))
print("simp is from " + str(WS_lt_eqm.index.min()) + " to " + str(WS_lt_eqm.index.max()))

obsh = onsite_data_eqm.to_xarray()
simh = y_predict_train_eqm.rename(columns={o_wsnode+'_Predicted': o_wsnode, o_wdnode+'_Predicted': o_wdnode}).to_xarray()
simp = WS_lt_eqm.to_xarray()


In [ ]:
variable = o_wsnode

WS_lt_eqm_adj = adjust(
    method="quantile_mapping",
    obs=obsh[variable],
    simh=simh[variable],
    simp=simp[variable],
    input_core_dims={"obs": "time1", "simh": "time2", "simp": "time3"},
    n_quantiles=60,
    kind="+",
).to_dataframe()

#postprocessing back to normal
WS_lt_eqm_adj.index = WS_lt_eqm_adj.index.tz_localize(time_zone_site)
WS_lt_eqm_adj.index.name = WS_lt.index.name
WS_lt_eqm_adj

### Wind direction

In [ ]:
variable = o_wdnode

WS_lt_eqm_adj_wd = adjust(
    method="quantile_mapping",
    obs=obsh[variable],
    simh=simh[variable],
    simp=simp[variable],
    input_core_dims={"obs": "time1", "simh": "time2", "simp": "time3"},
    n_quantiles=60,
    kind="+",
).to_dataframe()

#postprocessing back to normal
WS_lt_eqm_adj_wd.index = WS_lt_eqm_adj_wd.index.tz_localize(time_zone_site)
WS_lt_eqm_adj_wd.index.name = WS_lt.index.name
WS_lt_eqm_adj_wd

In [ ]:
#post process for validation
WS_lt_eqm_adj = WS_lt_eqm_adj.merge(WS_lt_eqm_adj_wd, left_index=True, right_index=True)

data_predict_train_eqm_adj = WS_lt_eqm_adj.loc[start_training:end_training,columnsY_ps].merge(onsite_data.loc[start_training:end_training,columnsY_ps],how='inner',left_index=True, right_index=True,suffixes= ('_Predicted', '_Original') )
data_predict_train_eqm_adj.dropna(inplace=True)
y_predict_train_eqm_adj = data_predict_train_eqm_adj.loc[start_training:end_training,[l + '_Predicted' for l in columnsY_ps]].copy()
y_predict_train_eqm_adj

In [ ]:
data_predict_test_eqm_adj = WS_lt_eqm_adj.loc[start_testing:end_testing,columnsY_ps].merge(onsite_data.loc[start_testing:end_testing,columnsY_ps],how='inner',left_index=True, right_index=True,suffixes= ('_Predicted', '_Original') )
data_predict_test_eqm_adj.dropna(inplace=True)
y_predict_test_eqm_adj = data_predict_test_eqm_adj.loc[start_testing:end_testing,[l + '_Predicted' for l in columnsY_ps]].copy()
y_predict_test_eqm_adj

## Scores

In [ ]:
#we give example of both error values using the previous baseline models
overall_score_best = overall_score(y_test[o_wsnode].to_numpy(), y_predict_test_eqm_adj[o_wsnode+'_Predicted'].to_numpy())
binned_error_best = binned_error(y_test[o_wsnode].to_numpy(), y_predict_test_eqm_adj[o_wsnode+'_Predicted'].to_numpy())
best_score_best = 100 * sqrt(mean_squared_error(y_test[o_wsnode], y_predict_test_eqm_adj[o_wsnode+'_Predicted'])) / y_test[o_wsnode].max()
print(f'Score is {best_score_best:.2f}% ')
print(f'Binned error is {100*binned_error_best:.2f}% ')
print(f'Overall error is  {100*overall_score_best:.2f}% ')

In [ ]:
#we give example of both error values using the previous baseline models
overall_score_best = overall_score(y_test[o_wdnode].to_numpy(), y_predict_test_eqm_adj[o_wdnode+'_Predicted'].to_numpy())
binned_error_best = binned_error(y_test[o_wdnode].to_numpy(), y_predict_test_eqm_adj[o_wdnode+'_Predicted'].to_numpy())
best_score_best = 100 * sqrt(mean_squared_error(y_test[o_wdnode], y_predict_test_eqm_adj[o_wdnode+'_Predicted'])) / y_test[o_wdnode].max()
print(f'Score is {best_score_best:.2f}% ')
print(f'Binned error is {100*binned_error_best:.2f}% ')
print(f'Overall error is  {100*overall_score_best:.2f}% ')

## Histogram check

In [ ]:
#Wind speed
fig, ax = plt.subplots(figsize=(18, 8))
bins_0 = np.arange(0.0, 30, 1)
ax.hist(onsite_data[o_wsnode],bins=bins_0, density = True, histtype='step', lw = 2, label='On-Site WS data (Training)')
ax.hist(y_predict_train_eqm_adj[o_wsnode+'_Predicted'],bins=bins_0, density = True, histtype='step', lw = 2, label='best_y_pred (training)')
ax.hist(WS_lt_eqm_adj[o_wsnode],bins=bins_0, density = True, histtype='step', lw = 2, label='y_pred_LT (long-term)')

handles, labels = ax.get_legend_handles_labels()
lgd = fig.legend(handles, labels)
#plt.ylim(0,0.00002)
plt.xlabel('Wind speed [m/s]')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Wind direction
fig, ax = plt.subplots(figsize=(18, 8))
bins_0 = np.arange(0.0, 360, 10)
ax.hist(onsite_data[o_wdnode],bins=bins_0, density = True, histtype='step', lw = 2, label='On-Site WD data (Training)')
ax.hist(y_predict_train_eqm_adj[o_wdnode+'_Predicted'],bins=bins_0, density = True, histtype='step', lw = 2, label='best_y_pred (training)')
ax.hist(WS_lt_eqm_adj[o_wdnode],bins=bins_0, density = True, histtype='step', lw = 2, label='y_pred_LT (long-term)')

handles, labels = ax.get_legend_handles_labels()
lgd = fig.legend(handles, labels)
#plt.ylim(0,0.00002)
plt.xlabel('Wind direction [°]')
plt.ylabel('Frequency')
plt.show()

## Scatterplot check

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(18, 8))

ylim = (0, 30)

# 1st plot: ws vs y_train
#r2_score is not equal to pearson squared for any fit that is non linear!
#r2_1 = r2_score(data_train['ws scaled'], data_train[o_wsnode])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_train[v_wsnode], data_train[o_wsnode])
r2_1 = r_value**2

ax[0].scatter(data_train[v_wsnode], data_train[o_wsnode], marker='.', alpha=0.5)
ax[0].set_xlabel('Vortex Wind Speed [m/s] (training period)')
ax[0].set_ylabel('Onsite Wind speed [m/s] (training period)')
ax[0].set_ylim(*ylim)
ax[0].set_title(f'R² = {r2_1:.2f}')

# 2nd plot: ws vs best_y_pred
#r2_2 = r2_score(data_predict_train.iloc[:,0], data_predict_train.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_train[o_wsnode+'_Original'], data_predict_train[o_wsnode+'_Predicted'])
r2_2 = r_value**2

ax[1].scatter(data_predict_train_eqm_adj[o_wsnode+'_Original'],data_predict_train_eqm_adj[o_wsnode+'_Predicted'], marker='.', alpha=0.5)
ax[1].set_xlabel('Onsite Wind Speed [m/s] (training period)')
ax[1].set_ylabel('Predicted wind speed [m/s] (training period)')
ax[1].set_ylim(*ylim)
ax[1].set_title(f'R² = {r2_2:.2f}')


# 3nd plot: test period
#r2_3 = r2_score(data_predict_test.iloc[:,0], data_predict_test.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_test[o_wsnode+'_Original'], data_predict_test[o_wsnode+'_Predicted'])
r2_3 = r_value**2

ax[2].scatter(data_predict_test_eqm_adj[o_wsnode+'_Original'], data_predict_test_eqm_adj[o_wsnode+'_Predicted'], marker='.', alpha=0.5)
ax[2].set_xlabel('Vortex Wind Speed [m/s] (testing period)')
ax[2].set_ylabel('Predicted wind speed [m/s] (testing period)')
ax[2].set_ylim(*ylim)
ax[2].set_title(f'R² = {r2_3:.2f}')


plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(18, 8))

ylim = (0, 360)

# 1st plot: ws vs y_train
#r2_score is not equal to pearson squared for any fit that is non linear!
#r2_1 = r2_score(data_train['ws scaled'], data_train[o_wdnode])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_train[v_wdnode], data_train[o_wdnode])
r2_1 = r_value**2

ax[0].scatter(data_train[v_wdnode], data_train[o_wdnode], marker='.', alpha=0.5)
ax[0].set_xlabel('Vortex wind direction [°] (training period)')
ax[0].set_ylabel('Onsite wind direction [°] (training period)')
ax[0].set_ylim(*ylim)
ax[0].set_title(f'R² = {r2_1:.2f}')

# 2nd plot: ws vs best_y_pred
#r2_2 = r2_score(data_predict_train.iloc[:,0], data_predict_train.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_train[o_wdnode+'_Original'], data_predict_train[o_wdnode+'_Predicted'])
r2_2 = r_value**2

ax[1].scatter(data_predict_train_eqm_adj[o_wdnode+'_Original'],data_predict_train_eqm_adj[o_wdnode+'_Predicted'], marker='.', alpha=0.5)
ax[1].set_xlabel('Onsite wind direction [°] (training period)')
ax[1].set_ylabel('Predicted wind direction [°] (training period)')
ax[1].set_ylim(*ylim)
ax[1].set_title(f'R² = {r2_2:.2f}')


# 3nd plot: test period
#r2_3 = r2_score(data_predict_test.iloc[:,0], data_predict_test.iloc[:,1])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(data_predict_test[o_wdnode+'_Original'], data_predict_test[o_wdnode+'_Predicted'])
r2_3 = r_value**2

ax[2].scatter(data_predict_test_eqm_adj[o_wdnode+'_Original'], data_predict_test_eqm_adj[o_wdnode+'_Predicted'], marker='.', alpha=0.5)
ax[2].set_xlabel('Vortex wind direction [°] (testing period)')
ax[2].set_ylabel('Predicted wind direction [°] (testing period)')
ax[2].set_ylim(*ylim)
ax[2].set_title(f'R² = {r2_3:.2f}')


plt.tight_layout()
plt.show()

## Monthly/hourly profiles

In [ ]:
# binning of original PTS
# Visualise monthly and hourly wind speed profiles

# Use only wind speed columns for plotting (e.g., o_wsnode for onsite, 'M(m/s)' for reference, and WS_lt for prediction)
WS_lt_month_eqm_adj = WS_lt_eqm_adj.groupby(WS_lt.index.month).mean()[o_wsnode]
WS_lt_hour_eqm_adj = WS_lt_eqm_adj.groupby(WS_lt.index.hour).mean()[o_wsnode]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(18, 8))

# Monthly profile
ax[0].plot(WS_lt_month.index, WS_lt_month.values, 'k-', label='Long Term Wind Speed (Prediction)')
ax[0].plot(WS_lt_month_eqm_adj.index, WS_lt_month_eqm_adj.values, 'g-', label='EQM Long Term Wind Speed (Prediction)')
ax[0].plot(onsite_month.index, onsite_month.values, 'r-', lw=2, label='On-Site WS data (Training)')
ax[0].set_xlabel('Month')
ax[0].set_xticks(np.arange(1, 13, 1))
ax[0].set_ylabel('Wind Speed [m/s]')
ax[0].set_title('Monthly Wind Speed Profile')

# Hourly profile
ax[1].plot(WS_lt_hour.index, WS_lt_hour.values, 'k-', label='Long Term Wind Speed (Prediction)')
ax[1].plot(WS_lt_hour_eqm_adj.index, WS_lt_hour_eqm_adj.values, 'g-', label='EQM Long Term Wind Speed (Prediction)')
ax[1].plot(onsite_hour.index, onsite_hour.values, 'r-', lw=2, label='On-Site WS data (Training)')
ax[1].set_xlabel('Hour of Day')
ax[1].set_xticks(np.arange(0, 24, 3))
ax[1].set_ylabel('Wind Speed [m/s]')
ax[1].set_title('Hourly Wind Speed Profile')

# Combine legends
handles1, labels1 = ax[0].get_legend_handles_labels()
plt.legend(handles1, labels1 , loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

## FFT

In [ ]:
# Number of samplepoints
N2 = len(onsite_data)
N = len(WS_lt_eqm_adj)

# sample spacing (in seconds)
T = 3600
# Calculate the FFT frequency step robustly
if WS_lt_eqm_adj.index.freq is not None:
    nb_sec_freq = WS_lt_eqm_adj.index.freq.delta.total_seconds()
else:
    # fallback: infer frequency from index if freq is None
    nb_sec_freq = (WS_lt_eqm_adj.index[1] - WS_lt_eqm_adj.index[0]).total_seconds()

#FFT of Synthesised data
y = WS_lt_eqm_adj.iloc[:,0]
yf = scipy.fftpack.fft(y.values)
xf = np.linspace(0.0, 1.0/(2.0*T), int(N/2))
xf_hour = 1/xf/3600/24/(nb_sec_freq/100)

#FFT of Measured data
y2 = onsite_data[o_wsnode]
y2 = y2.fillna(0) #Replacing NaN by 0s
yf2 = scipy.fftpack.fft(y2.values)
xf2 = np.linspace(0.0, 1.0/(2.0*T), int(N2/2))
xf_hour2 = 1/xf2/3600/24/6

#Plot
fig, ax = plt.subplots(figsize=(18, 10))
ax.plot(xf_hour, 2.0/N * np.abs(yf[:N//2]), color = 'black', label='Long Term PTS (Prediction)' )
ax.plot(xf_hour2, 2.0/N2 * np.abs(yf2[:N2//2]), color = 'red',linestyle=':', label='OYA PTS (Training)')
ax.set_xlim(0,10)
ax.set_xticks(np.arange(0, 10, step=1))
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles, labels)
ax.set_xlabel('Days')
ax.set_ylabel('Amplitude')

plt.show()

## Scaling factor

In [ ]:
print(f'Mean wind speed of onsite data during training period is {onsite_data.loc[start_training:end_training,o_wsnode].mean():.2f} m/s ')
print(f'Mean wind speed of predicted data during training period is {y_predict_train_eqm_adj.loc[start_training:end_training,o_wsnode+'_Predicted'].mean():.2f} m/s ')
print(f'Mean wind speed of predicted data during long-term period is {WS_lt_eqm_adj.loc[:,o_wsnode].mean():.2f} m/s ')
print(f'LT scaling factor is {100*((WS_lt_eqm_adj.loc[:,o_wsnode].mean()/y_predict_train_eqm_adj.loc[start_training:end_training,o_wsnode+'_Predicted'].mean())-1):.2f}% ')
